In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("kidney_disease.csv")

print(df.shape)
print(df.head())
print(df.info())

(400, 26)
   id   age    bp     sg   al   su     rbc        pc         pcc          ba  \
0   0  48.0  80.0  1.020  1.0  0.0     NaN    normal  notpresent  notpresent   
1   1   7.0  50.0  1.020  4.0  0.0     NaN    normal  notpresent  notpresent   
2   2  62.0  80.0  1.010  2.0  3.0  normal    normal  notpresent  notpresent   
3   3  48.0  70.0  1.005  4.0  0.0  normal  abnormal     present  notpresent   
4   4  51.0  80.0  1.010  2.0  0.0  normal    normal  notpresent  notpresent   

   ...  pcv    wc   rc  htn   dm  cad appet   pe  ane classification  
0  ...   44  7800  5.2  yes  yes   no  good   no   no            ckd  
1  ...   38  6000  NaN   no   no   no  good   no   no            ckd  
2  ...   31  7500  NaN   no  yes   no  poor   no  yes            ckd  
3  ...   32  6700  3.9  yes   no   no  poor  yes  yes            ckd  
4  ...   35  7300  4.6   no   no   no  good   no   no            ckd  

[5 rows x 26 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entrie

In [7]:
# CELL 4: Missing values check

df.isnull().sum()

id                  0
age                 9
bp                 12
sg                 47
al                 46
su                 49
rbc               152
pc                 65
pcc                 4
ba                  4
bgr                44
bu                 19
sc                 17
sod                87
pot                88
hemo               52
pcv                70
wc                105
rc                130
htn                 2
dm                  2
cad                 2
appet               1
pe                  1
ane                 1
classification      0
dtype: int64

In [8]:
# CELL 5: Target column distribution

df['classification'].value_counts()

classification
ckd       248
notckd    150
ckd\t       2
Name: count, dtype: int64

In [9]:
# CELL 6: Replace '?' values

df = df.replace('?', np.nan)

print("✅ '?' replaced with NaN")

✅ '?' replaced with NaN


In [10]:
# CELL 7: Strip spaces from object columns

df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

print("✅ Spaces cleaned")

✅ Spaces cleaned


In [11]:
# CELL 8: Convert numeric-looking columns

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='ignore')

print("✅ Numeric conversion attempted")

✅ Numeric conversion attempted


C:\Users\Admin\AppData\Local\Temp\ipykernel_21340\2958853649.py:4: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], errors='ignore')


In [12]:
# CELL 9: Numerical imputation

num_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print("✅ Numerical missing values handled")

✅ Numerical missing values handled


In [13]:
# CELL 10: Categorical imputation

cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

print("✅ Categorical missing values handled")

✅ Categorical missing values handled


In [14]:
# CELL 11: Encode categorical columns

from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("✅ Encoding completed")

✅ Encoding completed


In [15]:
# CELL 12: Final null check

print("Remaining null values:", df.isnull().sum().sum())

Remaining null values: 0


In [17]:
# NEW CELL: Synthetic dataset expansion to 20,000

import numpy as np

target_size = 20000

# sample with replacement
df_big = df.sample(n=target_size, replace=True, random_state=42).copy()

# add small noise to numerical columns
num_cols = df_big.select_dtypes(include=['float64', 'int64']).columns

for col in num_cols:
    std = df_big[col].std()
    if std != 0:
        noise = np.random.normal(0, 0.01 * std, size=target_size)
        df_big[col] = df_big[col] + noise

# save expanded dataset
df_big.to_csv("ckd_20000.csv", index=False)

print("✅ Expanded dataset created:", df_big.shape) 

✅ Expanded dataset created: (20000, 26)


In [19]:
# Load expanded dataset

df = pd.read_csv("ckd_20000.csv")

In [27]:
# 🔥 IMPORTANT FIX — drop id column if exists
if 'id' in df.columns:
    df = df.drop('id', axis=1)

In [20]:
# STEP B: Split features and target

X = df.drop("classification", axis=1)
y = df["classification"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (20000, 25)
y shape: (20000,)


In [21]:
# STEP C: Train-test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (16000, 25)
Test: (4000, 25)


In [22]:
# STEP D: Train model

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

print("✅ Model trained")

✅ Model trained


In [23]:
# STEP E: Prediction

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("✅ Prediction done")

✅ Prediction done


In [24]:
# STEP F: Evaluation

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0

Confusion Matrix:
 [[2496    0]
 [   0 1504]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      2496
           1       1.00      1.00      1.00      1504

    accuracy                           1.00      4000
   macro avg       1.00      1.00      1.00      4000
weighted avg       1.00      1.00      1.00      4000



In [26]:
# STEP G: Save model

import pickle

pickle.dump(model, open("ckd_model.pkl", "wb"))

print("✅ Model saved")

✅ Model saved
